<h2>Imports</h2>

In [ ]:
!pip install sentence-transformers nltk scikit-learn PyPDF2

import nltk
import glob
import re
import numpy as np
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files
from PyPDF2 import PdfReader

nltk.download('punkt')

<h2>Upload PDF</h2>

In [ ]:
uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]

print("Uploaded:", pdf_file)


In [ ]:

def extract_pdf_text(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

input_text = extract_pdf_text(pdf_file)

print("\nSample Text:\n", input_text[:300])



<h2>Pre processing</h2>

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    return text

input_text = preprocess(input_text)

<h2>Dataset</h2>

In [ ]:
dataset_path = "/content/drive/MyDrive/pan11"

source_files = glob.glob(dataset_path + "/**/source-document*.txt", recursive=True)

print("Total source files:", len(source_files))

source_files = source_files[:30]



<h2>Processing</h2>

In [ ]:

source_texts = []

for file in source_files:
    try:
        with open(file, 'r', encoding='latin-1') as f:
            source_texts.append(preprocess(f.read()))
    except:
        continue


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

input_sentences = sent_tokenize(input_text)

source_sentences = []

for src in source_texts:
    source_sentences.extend(sent_tokenize(src))

print("Total source sentences:", len(source_sentences))


In [ ]:

source_embeddings = model.encode(source_sentences, batch_size=64, show_progress_bar=True)
input_embeddings = model.encode(input_sentences, batch_size=64, show_progress_bar=True)

similarity_matrix = cosine_similarity(input_embeddings, source_embeddings)


results = []

for i, sent in enumerate(input_sentences):
    best_idx = np.argmax(similarity_matrix[i])
    best_score = similarity_matrix[i][best_idx]
    best_match = source_sentences[best_idx]

    results.append((sent, best_match, best_score))


<h2>Condition for plagiarism</h2>

In [ ]:
threshold = 0.5

plag = [r for r in results if r[2] > threshold]

plag_percent = (len(plag) / len(results)) * 100

print(f"\nPAN DATASET PLAGIARISM: {plag_percent:.2f}%")


<h2>Display Plagiarism</h2>

In [ ]:

if len(plag) > 0:
    print("\nDetected Plagiarized Sentences:\n")

    for sent, match, score in plag[:10]:
        print("Input:", sent)
        print("Match:", match)
        print("Score:", score)
        print("="*100)

else:
    print("\nNo strong plagiarism. Showing top matches:\n")

    results = sorted(results, key=lambda x: x[2], reverse=True)

    for sent, match, score in results[:5]:
        print("Input:", sent)
        print("Match:", match)
        print("Score:", score)
        print("="*100)